# APEX — Stage 1: Ingest

**Goal of this stage:** download every 2023 *qualifying* session from FastF1, stack all the laps into one table, save it, and eyeball what we got.

No cleaning or modelling yet — we just want the raw data in front of us.

### 1. Imports & cache
FastF1 talks to the official F1 timing servers. The **cache** means each session is downloaded once and read from disk forever after.

In [ ]:
import fastf1                       # the data source (official F1 timing)
import pandas as pd                 # tables (DataFrames)
from pathlib import Path            # OS-safe file paths


# Walk up the folder tree until we find WHITEPAPER.md, so this notebook works
# no matter which directory it's run from.
def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    while p != p.parent:                     # stop at the filesystem root
        if (p / "WHITEPAPER.md").exists():
            return p
        p = p.parent
    raise RuntimeError("repo root not found")


REPO = find_repo_root(Path.cwd())
CACHE = REPO / "data" / "cache"         # FastF1's download cache
PARQUET = REPO / "data" / "parquet"     # where we save our own tables
CACHE.mkdir(parents=True, exist_ok=True)
PARQUET.mkdir(parents=True, exist_ok=True)

fastf1.Cache.enable_cache(str(CACHE))   # first run downloads; later runs read locally
print("Repo root:", REPO)

### 2. The season schedule
Every race weekend in 2023. `include_testing=False` drops pre-season testing (not competitive sessions).

In [ ]:
SEASON = 2023

schedule = fastf1.get_event_schedule(SEASON, include_testing=False)
schedule[["RoundNumber", "EventName", "Country", "EventDate"]]

### 3. Pull every qualifying session
For each round we load the `'Q'` session and copy out its lap table. We load laps + weather + race-control messages but **not telemetry** (the per-metre car-sensor data) — we don't need it and it's ~100× larger. Each lap is tagged with its season/round/event so we never lose track of which session it came from.

In [ ]:
frames = []
for rnd, name in zip(schedule["RoundNumber"], schedule["EventName"]):
    session = fastf1.get_session(SEASON, rnd, "Q")   # the qualifying session object
    session.load(telemetry=False)                    # downloads on first run, cached after
    laps = session.laps.copy()                       # this session's lap table
    laps["Season"] = SEASON                           # tag so we can tell sessions apart
    laps["Round"] = rnd
    laps["EventName"] = name
    frames.append(laps)
    print(f"R{rnd:>2}  {name:<30} {len(laps):>4} laps")

# Stack all sessions into one plain DataFrame (one row per lap).
laps_raw = pd.DataFrame(pd.concat(frames, ignore_index=True))
print("\nTOTAL:", laps_raw.shape[0], "laps x", laps_raw.shape[1], "columns")

### 4. Save the raw laps
Write to Parquet (a fast, compact table format) so we can experiment later without re-downloading.

In [ ]:
out_path = PARQUET / f"laps_q_{SEASON}_raw.parquet"
laps_raw.to_parquet(out_path)
print("Saved:", out_path.relative_to(REPO))

### 5. Eyeball what we got
A few key columns for the first handful of laps. `LapTime` is a duration; `TrackStatus`, `Deleted` and `IsPersonalBest` are the flags we'll use to clean the data in the next stage.

In [ ]:
cols = ["EventName", "Driver", "Team", "LapNumber", "LapTime",
        "Compound", "TrackStatus", "Deleted", "IsPersonalBest"]
laps_raw[cols].head(12)

---
## APEX — Stage 2: Cleaning

**Goal:** keep only *genuine flying laps* — laps that truly reflect qualifying pace — and drop everything else, then slim the table to the columns we need.

A real qualifying lap is: a completed timed lap, **not** an in-/out-lap, **not** deleted, run under a **fully green** track, and marked **reliable** by FastF1. Below we test each condition in turn and print how many laps drop out at each step, so nothing is hidden. (Runs from the raw file saved in Stage 1.)

In [ ]:
# Load the raw laps saved in Stage 1 (so this stage stands on its own).
raw = pd.read_parquet(PARQUET / f"laps_q_{SEASON}_raw.parquet")

# Keep a lap only if it passes EVERY test. We apply them one at a time and
# report how many laps each removes — the cleaning is fully transparent.
laps = raw.copy()


def keep(mask, reason):
    global laps
    before = len(laps)
    laps = laps[mask].copy()
    print(f"  drop {before - len(laps):>5,}  ({reason})  ->  {len(laps):,} left")


print(f"Start: {len(laps):,} laps")
keep(laps["LapTime"].notna(),          "no lap time  (out-lap / aborted run)")
keep(laps["PitOutTime"].isna(),        "out-lap  (just left the pits)")
keep(laps["PitInTime"].isna(),         "in-lap  (returning to the pits)")
keep(~laps["Deleted"].fillna(False),   "deleted  (e.g. track limits)")
keep(laps["TrackStatus"] == "1",       "not fully green  (yellow / SC / red)")
keep(laps["IsAccurate"].fillna(False), "flagged unreliable by FastF1")
print(f"\nClean flying laps: {len(laps):,}")

### Keep only the columns we need
The raw table has ~34 columns; most (speed traps, timestamps, positions) don't bear on pace. We select the ~10 that matter, add a human-readable `LapTimeSeconds`, and save to a **separate** `..._clean.parquet` — the raw file stays untouched as our source of truth.

In [ ]:
keep_cols = [
    "Season", "Round", "EventName",   # which session (tells us the circuit)
    "Driver", "Team",                  # who drove + which car
    "LapNumber", "LapTime",            # the lap and its raw pace
    "Compound", "TyreLife",            # tyre context (sanity checks)
    "IsPersonalBest",                  # sanity flag
]
clean = laps[keep_cols].reset_index(drop=True)

# LapTime is a duration; add a plain seconds column for easy reading/plotting.
clean["LapTimeSeconds"] = clean["LapTime"].dt.total_seconds()

clean_path = PARQUET / f"laps_q_{SEASON}_clean.parquet"
clean.to_parquet(clean_path)
print(f"{len(raw):,} raw  ->  {len(clean):,} clean  ({len(clean)/len(raw):.0%} kept)")
print("Saved:", clean_path.relative_to(REPO))
clean.head()

### Eyeball the result
Two quick checks that the cleaning behaved sensibly.

In [ ]:
# Check 1: clean laps per driver (~150-190 across 22 sessions is expected).
print(clean.groupby("Driver").size().sort_values(ascending=False).head())

# Check 2: each driver's fastest clean lap. These are RAW seconds, so the quickest
# are all from the SHORTEST circuit (Austria) — which is exactly why the next stage
# normalises pace to a %-gap, so laps from different circuits become comparable.
best = clean.loc[clean.groupby("Driver")["LapTimeSeconds"].idxmin()]
best[["Driver", "Team", "EventName", "LapTimeSeconds"]].sort_values("LapTimeSeconds").head(10)

---
## APEX — Stage 3: Normalisation

**Goal:** make laps from different circuits comparable. A 1:12 at Monaco and a 1:32 at Monza mean nothing side-by-side in raw seconds, so within **each session** we take the fastest lap as the benchmark and express every lap as a **% gap** to it:

`PacePct = 100 × (lap − session_best) / session_best`

The fastest lap of a session is `0.0`; everyone else is a positive % *slower*. Now a 0.3% gap means the same thing at every track — which is what lets us compare and model across the whole calendar.

In [ ]:
# Start from the cleaned laps (Stage 2).
clean = pd.read_parquet(PARQUET / f"laps_q_{SEASON}_clean.parquet")

# The fastest lap in each session (one session = one Round). transform("min")
# broadcasts that single value back onto every lap in the same session.
clean["SessionBest"] = clean.groupby(["Season", "Round"])["LapTimeSeconds"].transform("min")

# Percentage slower than the session's best lap. 0.0 == fastest lap of the session.
clean["PacePct"] = 100 * (clean["LapTimeSeconds"] - clean["SessionBest"]) / clean["SessionBest"]

norm_path = PARQUET / f"laps_q_{SEASON}_normalised.parquet"
clean.to_parquet(norm_path)
print("Saved:", norm_path.relative_to(REPO), "|", len(clean), "laps")
clean[["EventName", "Driver", "Team", "LapTimeSeconds", "SessionBest", "PacePct"]].head()

### Eyeball the result
Check the maths (each session's own best lap should read 0.0), then look at the spread.

In [ ]:
# Correctness: every session's minimum PacePct must be exactly 0 (its own best lap).
print("Max of per-session minimums:",
      round(clean.groupby("Round")["PacePct"].min().max(), 6), "(should be 0.0)")

# Distribution of the gap-to-best across all laps (note the large upper tail):
print(clean["PacePct"].describe()[["min", "25%", "50%", "75%", "max"]].round(2).to_string())

# Real example — Monaco 2023, the six quickest laps (VER pole, Alonso P2, Leclerc P3):
mon = clean[clean["EventName"] == "Monaco Grand Prix"].nsmallest(6, "PacePct")
mon[["Driver", "Team", "LapTimeSeconds", "PacePct"]]

> **Note — a tail of slow laps remains.** The median gap is ~2.9%, but the top quartile reaches ~19% and the max ~107% (a lap twice as slow as pole). These are cool-down / preparation laps that slipped through Stage 2 — they aren't in-/out-laps, deleted, or under a flag, so the filters kept them. They would distort the model, so the **next step** is to trim them (e.g. keep only laps within ~107% of the session best, or each driver's representative fast laps). Normalisation itself is correct; this is a cleaning refinement for later.

---
## APEX — Stage 3b: Trim to representative laps

Stage 3's note flagged a tail of very slow laps (cool-down / preparation laps, up to ~107% off pole) that survived cleaning. Here we remove them with the **107% rule**: keep only laps within **7% of the session's best lap** — F1's own benchmark for a competitive lap.

The fastest lap of each session is unchanged, so the `PacePct` values from Stage 3 stay valid — we're simply dropping the rows above the threshold.

In [ ]:
MAX_PCT = 7.0   # 107% rule: keep only laps within 7% of the session's best lap

norm = pd.read_parquet(PARQUET / f"laps_q_{SEASON}_normalised.parquet")
model = norm[norm["PacePct"] <= MAX_PCT].reset_index(drop=True)

model_path = PARQUET / f"laps_q_{SEASON}_model.parquet"
model.to_parquet(model_path)
print(f"{len(norm):,} normalised -> {len(model):,} representative ({len(model)/len(norm):.0%} kept)")
print("Saved:", model_path.relative_to(REPO))

# The spread is now sensible: ~1.9% median gap to pole, nothing beyond ~7%.
model["PacePct"].describe()[["min", "25%", "50%", "75%", "max"]].round(2)

### The model-ready dataset
`laps_q_2023_model.parquet` is what Stage 4 will consume: **1,902 representative qualifying laps**, each expressed as a % gap to session pole. The spread now runs 0–7% with a ~1.9% median — clean, comparable pace across all 22 circuits.

*(Stage 4 may aggregate further — e.g. each driver's best lap per session — but these representative laps are the foundation.)*

---
## APEX — Stage 4: Fitting the model (part 1 — the transparent version)

We split each lap's `PacePct` into two hidden pieces plus noise:

`PacePct  =  car-weekend effect  +  driver effect  +  noise`

- **Car-weekend effect** — one adjustment per team per round (`TeamSession`), absorbing how good that car was at that track that weekend.
- **Driver effect** — whatever pace is left once the car is accounted for.

Because teammates share a `TeamSession`, the driver effect is pinned down by the gap *between teammates in the same car* — the controlled experiment at the heart of APEX. Ordinary least-squares (OLS) fits all of these effects simultaneously, which is equivalent to chaining every teammate comparison together.

*(This is the simple, fully-transparent version; the mixed-effects upgrade — shrinkage + confidence intervals — is part 2.)*

In [ ]:
import statsmodels.formula.api as smf

m = pd.read_parquet(PARQUET / f"laps_q_{SEASON}_model.parquet")
m["TeamSession"] = m["Team"] + " R" + m["Round"].astype(str)   # one label per car-weekend

# Least squares fits every car-weekend effect and every driver effect at once.
fit = smf.ols("PacePct ~ C(TeamSession) + C(Driver)", data=m).fit()
print(f"laps = {int(fit.nobs)}   R^2 = {fit.rsquared:.2f}")

### The catch: one season can't rank the whole grid
If you print the raw driver coefficients, the *between-team* numbers explode to nonsense. That isn't a bug — it's the identification limit from the white paper (§2.3). To place every driver on one scale, the teams must be **linked** by drivers who moved between them. In 2023 essentially none did, so each team is an **isolated island**: we can compare teammates perfectly, but not one team against another.

So the trustworthy output from a single season is the **teammate battle** — each driver versus whoever shared their car.

In [ ]:
# Confirm the "islands": how many drivers raced for more than one team in 2023?
print("Drivers who raced for >1 team in 2023:",
      int((m.groupby("Driver")["Team"].nunique() > 1).sum()), "\n")

# The identified signal: each driver's average pace vs their car-weekend mean
# (negative = faster than whoever shared the car). Only comparable WITHIN a team.
m["dev"] = m["PacePct"] - m.groupby("TeamSession")["PacePct"].transform("mean")
battle = (m.groupby(["Team", "Driver"])
            .agg(laps=("dev", "size"), dev=("dev", "mean"))
            .reset_index()
            .sort_values(["Team", "dev"]))
for team, g in battle.groupby("Team"):
    parts = "   ".join(f"{r.Driver} {r.dev:+.3f}" for r in g.itertuples())
    print(f"  {team:<18} {parts}")

### Does it match reality? Yes.
- **Red Bull:** VER crushes PER (~0.79%/lap) — the biggest gap on the grid, exactly as in 2023.
- **Aston Martin** (ALO ≫ STR), **Williams** (ALB ≫ SAR), **Haas** (HUL > MAG) all line up.
- **Mercedes** (HAM ≈ RUS) and **Alpine** (OCO ≈ GAS) are near-dead-even — also correct.

Every battle matches what we saw in 2023, which validates the whole pipeline.

**Next (Stage 4, part 2):** to answer *"who is fastest across the whole grid"* we need the islands connected — that means **(a) multiple seasons**, so driver transfers link the teams into one network, and **(b) a mixed-effects model** that adds *shrinkage* (a driver with few laps isn't over-rated) and proper *confidence intervals*.

---
## APEX — Stage 4.2: Connecting the islands

Stage 4.1 showed a single season is *islands* — teammates compare cleanly, but teams
never link, so the grid can't be ranked end-to-end. Two things fix that, exactly as the
white paper (§2.3) predicts:

1. **More seasons.** Across **2018–2024**, drivers transfer between teams (Sainz
   McLaren→Ferrari, Albon Red Bull→Williams, Vettel Ferrari→Aston Martin…). Each transfer
   is a wire soldering two teams together; chain enough and the whole grid becomes one
   network. *(This is why the window must reach back to 2018 — Red Bull and Ferrari had
   such stable line-ups that 2021–2023 alone leaves them stranded.)*
2. **Shrinkage.** We fit the two-way effects model by **ridge-penalised** least squares:
   driver and car effects are pulled toward the field mean until the data earns a stronger
   claim (a rookie with 30 laps can't post a wild rating). Circuit stays a fixed baseline.

The heavy lifting now lives in the reusable `analysis/apex` package; this notebook just
narrates it.

In [ ]:
import sys
sys.path.insert(0, str(REPO))
import pandas as pd
from analysis.apex import SEASONS
from analysis.apex.data import build_model_dataset, model_path
from analysis.apex import model as M, views as V

# Build the 2018-2024 model dataset if it isn't cached yet (downloads once).
path = model_path(SEASONS)
if not path.exists():
    build_model_dataset(SEASONS)
m = pd.read_parquet(path)
print(f"{len(m):,} representative laps | seasons {sorted(m.Season.unique())} | "
      f"{m.Driver.nunique()} drivers")

# Connectivity: the identification network across all seasons.
net = M.build_network(m)
print(f"connected components: {len(net.components)}  |  rated drivers: {len(net.largest)}"
      f"  |  unrated: {net.unrated or 'none'}")

# Fit the two-way effects model (ridge lambda chosen by leave-session-out CV).
fit = M.fit_model(m, verbose=False)
print(f"\nmodel: {len(m):,} laps, R^2={fit.r2:.2f}, lambda={fit.lam}\n")
print("Driver skill — top 10 (higher = faster, % pace vs field average):")
for c in sorted(fit.skill, key=fit.skill.get, reverse=True)[:10]:
    print(f"  {c:>4} {fit.skill[c]:+.3f}")

### Connected — and the teammate battles still hold

One component, no unrated drivers: the whole grid is finally comparable. The multi-season
fit also preserves the within-team gaps validated in Stage 4.1 (VER ≫ PER, ALO ≫ STR,
HAM ≈ RUS) — the global scale is bolted onto battles we already trust, not invented.

Read the car effects carefully: they are pace **net of driver**. Red Bull 2023 does not
top the car table because the model credits much of that car's dominance to Verstappen —
which is exactly the confounding APEX exists to separate.

---
## APEX — Stage 5: The dashboard views

With skill, car performance and their uncertainty in hand, the rest is bookkeeping — the
three flagship views plus the two nice-to-haves, all read off the same fit:

- **Equalise** — drop everyone into one car; the finishing order collapses to skill alone.
- **Heatmap** — each driver's pace at each circuit vs their own baseline (the residual).
- **Career arc** — per-season skill as a shrunk deviation off the career anchor.

In [ ]:
bs = M.bootstrap(fit, m, n_reps=100, verbose=False)   # session-level confidence
eq = V.equalise(m, fit, bs)
print(f"EQUALISE — everyone in one car, reference {eq['reference_circuit']} "
      f"({eq['reference_lap_seconds']:.2f}s). Top 8 by pure skill:")
for o in eq["order"][:8]:
    print(f"  {o['code']:>4}  +{o['gap_seconds']:.3f}s")

hm = V.heatmap(m, fit)
print(f"\nHEATMAP — {len(hm['drivers'])} drivers x {len(hm['circuits'])} circuits "
      f"({hm['unit']}); negative = overperformance.")

arc = M.fit_arc(fit, m, n_reps=80, verbose=False)
print(f"\nCAREER ARC — {len(arc)} drivers; Verstappen season by season:")
for pt in arc["VER"]:
    print(f"  {pt['season']}  skill {pt['skill']:+.3f}")

---
## APEX — Stage 6: Ship the artifact

Everything the dashboard needs is precomputed into one compact JSON (< 1 MB): driver
skill and car performance with confidence bands, the teammate network, the equalised
grid, the driver×circuit heatmap, career arcs, and the driver covariance for
head-to-head. The dashboard fetches it client-side — the model never runs at request
time.

`analysis/build_artifact.py` runs this whole chain headless; the contract is documented
in `analysis/SCHEMA.md`.

In [ ]:
from analysis.apex.build import build, write

# Full chain -> one compact JSON. 300 bootstrap reps for the shipped confidence bands.
artifact = build(SEASONS, n_reps=300, verbose=False)
write(artifact)

print("keys:", list(artifact))
print("meta:", {k: artifact["meta"][k] for k in
      ["seasons", "n_laps", "n_drivers", "n_car_seasons", "r2", "connected"]})